In [27]:
# Import the OpenAI class from the OpenAI Python library.
# WHY: We use this class to create an API client that communicates
# with an AI model through an OpenAI-compatible API.

from openai import OpenAI

# Import Python's built-in os module.
# WHY: It allows us to access environment variables,
# including API keys stored outside our Python code.

import os

# Import load_dotenv from the python-dotenv library.
# WHY: It loads variables from our .env file
# into the environment so Python can access them.

from dotenv import load_dotenv

# Import our custom website scraping function.
# WHY: fetch_website_contents(url) visits a website,
# extracts its content, and returns it as text.
#
# NOTE: This function must exist in scraper.py.

from scraper import fetch_website_contents

# IMPORT MARKDOWN DISPLAY TOOLS
# Import two utilities from IPython.
#
# Markdown:
#   Converts Markdown-formatted text into an object
#   that Jupyter Notebook can render.
#
# display:
#   Displays that object as formatted output.
#
# --------------------------------------------------

from IPython.display import Markdown, display

In [26]:
# --------------------------------------------------
# STEP 1: LOAD OUR SECRET API KEY
# --------------------------------------------------

# Load environment variables from the .env file.
# WHY: We should not write secret API keys directly
# inside our Python code.

load_dotenv()

# Get the API key stored under the name GROQ_API_KEY.
# WHY: The API provider uses this key to authenticate us.

api_key = os.environ.get("GROQ_API_KEY")

# Check whether the API key exists.
# WHY: Without an API key, our API request cannot work.
# If it is missing, stop the program with an error.

if not api_key:
    raise ValueError(
        "GROK_API_KEY environment variable is not set."
    )



In [14]:
# --------------------------------------------------
# STEP 2: CREATE OUR AI CLIENT
# --------------------------------------------------

# Create an OpenAI client configured for our API provider.
#
# WHY:
# The OpenAI Python library can communicate with
# other providers that support the OpenAI API format.
#
# api_key:
#   Our secret key used for authentication.
#
# base_url:
#   The API endpoint where requests will be sent.
#
# IMPORTANT:
#   The base_url must match the actual provider's API.
#   For xAI's Grok API, the standard endpoint is
#   https://api.x.ai/v1
#
#   The URL below is preserved from your original code
#   but should be corrected if you are using xAI Grok.

client = OpenAI(
    api_key=api_key,
    base_url="https://api.groq.com/openai/v1"
)

In [15]:
# --------------------------------------------------
# STEP 3: DEFINE THE SYSTEM PROMPT
# --------------------------------------------------

# The system prompt defines how the AI should behave.
#
# WHY:
# It tells the model its role, writing style,
# response format, and general instructions.
#
# In this case, the AI should:
# 1. Analyze website content.
# 2. Produce a short summary.
# 3. Use a humorous and snarky tone.
# 4. Ignore navigation-related text.
# 5. Respond in Markdown format.
#
# This prompt is reused whenever we summarize a website.

system_prompt = """
You are a snarky assistant that analyzes the contents of a website,
and provides a short, snarky, humorous summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""


# --------------------------------------------------
# STEP 4: DEFINE THE USER PROMPT PREFIX
# --------------------------------------------------

# This is the instruction we send along with
# the actual contents of a website.
#
# WHY:
# The system prompt defines the AI's behavior.
# The user prompt tells it what task to perform.
#
# Later, we will combine this prefix with the
# website text extracted by our scraper.
#
# Example:
#
# user_prompt_prefix + website
#
# Result:
#
# "Here are the contents of a website...
#  Provide a short summary...
#  [Actual website content]"

user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.

"""

In [16]:
# --------------------------------------------------
# STEP 5: BUILD THE MESSAGES FOR THE AI
# --------------------------------------------------

# Define a function named message_for.
#
# WHY:
# Instead of manually creating messages every time,
# we use a reusable function.
#
# Parameter:
#   website = the extracted website content (text).
#
# This function returns a list containing two messages:
#
# 1. System message:
#    Tells the AI how to behave.
#
# 2. User message:
#    Contains our instructions + website contents.

def message_for(website):

    return [
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": user_prompt_prefix + website
        }
    ]


In [17]:
# --------------------------------------------------
# STEP 6: CREATE THE WEBSITE SUMMARY FUNCTION
# --------------------------------------------------

# Define a function named summarize_website.
#
# WHY:
# We want one reusable function that:
#
# 1. Takes a website URL.
# 2. Extracts the website content.
# 3. Prepares messages for the AI.
# 4. Sends those messages to the AI model.
# 5. Returns the generated summary.
#
# Parameter:
#   url = the website address we want to summarize.

def summarize_website(url):

    # STEP 6.1: FETCH WEBSITE CONTENT
    # Example:
    # url = "https://example.com"

    website = fetch_website_contents(url)
    
    # STEP 6.2: SEND WEBSITE CONTENT TO THE AI
    #
    # client.chat.completions.create():
    # Sends our messages to the configured AI API.
    #
    # model:
    #   Identifies which AI model to use.
    #   The model name must be supported by the provider.
    #
    # messages:
    #   message_for(website) builds the message list.
    #
    # The returned response contains the model's output
    # along with additional response information.

    response = client.chat.completions.create(

        model="openai/gpt-oss-20b",

        messages=message_for(website)
    )


    # STEP 6.3: EXTRACT THE AI-GENERATED SUMMARY
    #
    # response:
    #   The complete response returned by the API.
    #
    # choices:
    #   A list of generated response choices.
    #
    # [0]:
    #   Selects the first response choice.
    #
    # message:
    #   The assistant message inside that choice.
    #
    # content:
    #   The actual text generated by the AI.
    #
    # return:
    #   Sends the summary back to whoever called
    #   summarize_website(url).
    #
    # IMPORTANT:
    # return does not print anything.
    # It only returns the result.

    return response.choices[0].message.content


In [19]:
# --------------------------------------------------
# STEP 7: EXAMPLE OF HOW TO USE THE FUNCTION
# --------------------------------------------------

summarize_website("https://edwarddonner.com")

# This would:
# 1. Fetch the website text.
# 2. Send it to the AI.
# 3. Receive a summary.
# 4. Print the summary on our screen.

'**Edward Donner’s personal “hub of genius”**  \n\n- *Who is Ed?* A former JPMorgan MD, ex‑CEO of a now‑acquired AI startup, and current co‑founder/CTO of Nebula.io. He’s also a wannabe music producer and a self‑appointed “LLM guru.”  \n- *What does the site do?*  \n  - **Curriculum & Udemy** – He turned his endless rambling about LLMs into bestselling courses (1\u202fM+ enrollments, 194 countries).  \n  - **Digital Avatar** – A chatbot so eager it’s practically a second personality.  \n  - **Outsmart** – An arena where LLMs duke it out in “diplomacy and deviousness.”  \n- *News & Posts (because he can’t keep quiet)*  \n  - **Oct\u202f14\u202f2026** – “AI FDE: The Forward Deployed Engineer & Agentic AI Architect.”  \n  - **Feb\u202f17\u202f2026** – “AI Coder: Vibe Coder to Agentic Engineer – RESOURCES.”  \n  - **Jan\u202f4\u202f2026** – “AI Builder with n8n – Create Agents and Voice Agents – RESOURCES.”  \n  - **Sept\u202f15\u202f2025** – “AI Engineering MLOps Track – Deploy AI to Prod

In [30]:
# A function to display this nicely in the output, using markdown

def display_summary(url):
    summary = summarize_website(url)
    # Convert the summary text into a Markdown object
    # and display it in a formatted way.
    #
    # Markdown(summary):
    #   Tells Jupyter Notebook to interpret the text
    #   as Markdown instead of ordinary plain text.
    #
    # display():
    #   Renders the Markdown object in the notebook.
    #
    # Example:
    # "**Hello**" will appear as bold Hello.
    display(Markdown(summary))

In [29]:
display_summary("https://www.thedailystar.net/")

**The Daily Star, Bangladesh – because *every* day needs a reminder that the world is still happening.**  
- **Crime headline:** “Ondhokarer Golpo” – a nostalgic throw‑back to the ‘90s when two brothers were the real reason Dhaka’s underworld was scared of being scared.  
- **Education chaos:** 4 lakh students are slated to retake exams in a month. Boards, we hope you’re ready to juggle that paperwork.  
- **Finance 101:** “How to make shariah‑based investments?” – a guide for those who want to put their money in something other than the usual stock market.  

The rest of the page is a sprawling list of categories that basically says, “We cover everything, from politics to pop culture, but we’re probably not actually reading your comment.”